# HAM10000 – Tiền xử lý dữ liệu (dạng bảng) & Baseline ResNet-18

Notebook này:
1. Đọc metadata + ánh xạ đường dẫn ảnh
2. **Kiểm tra & tiền xử lý dữ liệu dạng bảng** (không xử lý ảnh): giá trị thiếu, trùng lặp, tính nhất quán, outlier, phân bố nhãn
3. Mã hoá nhãn, chia **train / val / test theo `lesion_id`**
4. **Xử lý mất cân bằng lớp bằng Weighted Random Sampling** (`WeightedRandomSampler`)
5. Huấn luyện **baseline ResNet-18** (chỉ resize + normalize, không xử lý ảnh chương 3) với **3 seed** và ghi lại điểm

Mọi kết quả lưu ở `outputs/baseline/` (số liệu) và `outputs/figures/baseline/` (hình).

## 1. Import & cấu hình

In [ ]:
import copy
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             precision_recall_fscore_support, confusion_matrix)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms

from src.eda.plots import CLASS_COLORS, setup_style   # cùng màu/thứ tự lớp với EDA

SEED = 42
DATA_DIR = Path("archive")
METADATA_CSV = DATA_DIR / "HAM10000_metadata.csv"
IMAGE_DIRS = [DATA_DIR / "HAM10000_images_part_1", DATA_DIR / "HAM10000_images_part_2"]
SPLIT_DIR = Path("outputs/splits")
PREP_DIR = Path("outputs/data_prep")
OUT_DIR = Path("outputs/baseline")
FIG_DIR = Path("outputs/figures/baseline")
for d in (SPLIT_DIR, PREP_DIR, OUT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

TEST_SIZE = 0.2      # ~20% test (đã cố định trong outputs/splits/test.csv)
VAL_SIZE = 0.2       # ~20% của train -> validation (chọn epoch, không dùng test)
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0      # Windows + Jupyter: để 0 cho an toàn

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ưu tiên tái lập hơn tốc độ
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
setup_style()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")

## 2. Đọc dữ liệu

In [ ]:
df = pd.read_csv(METADATA_CSV)
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# Ánh xạ image_id -> đường dẫn file (ảnh nằm rải ở 2 thư mục)
image_paths = {p.stem: p for d in IMAGE_DIRS for p in d.glob("*.jpg")}
df["path"] = df["image_id"].map(image_paths)
print("Số file ảnh tìm thấy:", len(image_paths))

## 3. Kiểm tra & tiền xử lý dữ liệu dạng bảng

Phần này **chỉ xử lý các vấn đề về số liệu/metadata**, không đụng tới pixel ảnh. Tóm tắt cách làm cho từng vấn đề
(số liệu cụ thể xem output các ô bên dưới):

| Vấn đề | Cách kiểm tra | Cách xử lý đang dùng | Lý do |
|---|---|---|---|
| Giá trị thiếu (`age` NaN, `sex`/`localization` = `unknown`) | Đếm NaN + chuỗi `unknown`, xem phân bố theo `dx` | **Giữ toàn bộ dòng.** `age`: điền **median của tập train** + cột cờ `age_missing`. `sex`/`localization`: giữ `unknown` như một nhóm riêng | Mô hình chỉ dùng ảnh + `dx`; ảnh và nhãn của các dòng này đầy đủ → xoá dòng chỉ làm mất ảnh. Median fit trên train để không rò rỉ |
| Thiếu nhãn / thiếu file ảnh | `dx` NaN, `image_id` không có file | Dừng (assert) nếu có | Không có nhãn/ảnh thì không thể dùng |
| Trùng lặp | Dòng trùng, `image_id` trùng, file thừa; cặp ảnh gần trùng (từ EDA) | Không xoá; **chia theo `lesion_id`** | Ảnh gần trùng đều cùng `lesion_id` → nhóm theo lesion đã ngăn rò rỉ train/test |
| Nhất quán | 1 lesion ↔ 1 `dx`; metadata trong cùng lesion; kích thước/mode **toàn bộ** ảnh | Báo cáo; dừng nếu 1 lesion có nhiều `dx` | Nhãn mâu thuẫn trong một lesion sẽ là nhiễu nhãn |
| Outlier | Quy tắc IQR (1.5×IQR) cho `age`; số ảnh/lesion | **Gắn cờ, không xoá** (`age_outlier`) | Tuổi thấp vẫn hợp lý về mặt y khoa và không phải đầu vào mô hình; xoá sẽ mất ảnh hợp lệ |
| Mất cân bằng lớp | Tỉ lệ lớn nhất/nhỏ nhất | **Weighted Random Sampling** (mục 6) | Theo chỉ định; chi tiết ở mục 6 |

### 3.1 Giá trị thiếu (NaN và giá trị `unknown`)

In [ ]:
missing = pd.DataFrame({
    "nan": df.isna().sum(),
    "unknown": (df.astype(str).apply(lambda s: s.str.lower()) == "unknown").sum(),
})
missing["tổng"] = missing.sum(axis=1)
missing["%"] = (missing["tổng"] / len(df) * 100).round(2)
missing

In [ ]:
# Giá trị thiếu tập trung ở lớp nào? (nếu chỉ ở 1-2 lớp thì xoá dòng sẽ làm lệch phân bố nhãn)
miss_by_dx = pd.DataFrame({
    "age NaN": df[df["age"].isna()]["dx"].value_counts(),
    "sex unknown": df[df["sex"] == "unknown"]["dx"].value_counts(),
    "localization unknown": df[df["localization"] == "unknown"]["dx"].value_counts(),
    "tổng ảnh": df["dx"].value_counts(),
}).fillna(0).astype(int)
print("Số dòng thiếu ≥1 trường metadata:",
      (df["age"].isna() | (df["sex"] == "unknown") | (df["localization"] == "unknown")).sum())
miss_by_dx

> Giá trị thiếu chỉ nằm ở **metadata phụ** (`age`, `sex`, `localization`), không nằm ở ảnh hay nhãn `dx`.
> Xoá các dòng này sẽ bỏ ảnh hợp lệ và lấy bớt ảnh không đều giữa các lớp (xem bảng trên), nên **giữ lại tất cả dòng**.
> `age` sẽ được điền bằng median **tính trên tập train** sau khi chia (mục 5.3) và kèm cờ `age_missing`;
> `sex`/`localization` giữ `unknown` làm một nhóm riêng (điền mode sẽ tạo thông tin giả).

In [ ]:
assert df["dx"].isna().sum() == 0, "Có ảnh thiếu nhãn!"
assert df["path"].isna().sum() == 0, "Có image_id không tìm thấy file ảnh!"
df["age_missing"] = df["age"].isna()
print("Không thiếu nhãn, mọi image_id đều có file ảnh.")

### 3.2 Trùng lặp

In [ ]:
print("Dòng trùng hoàn toàn       :", df.drop(columns="path").duplicated().sum())
print("image_id trùng             :", df["image_id"].duplicated().sum())
orphans = set(image_paths) - set(df["image_id"])
print("File ảnh không có metadata :", len(orphans))

# Mỗi lesion_id có thể có nhiều ảnh -> phải chia theo nhóm
imgs_per_lesion = df.groupby("lesion_id").size()
print("Số lesion_id               :", df["lesion_id"].nunique())
print("Số ảnh / lesion            :", imgs_per_lesion.value_counts().sort_index().to_dict())

# Ảnh gần trùng (tính trong EDA, eda_image_quality.ipynb): có cặp nào khác lesion không?
nd_path = Path("outputs/eda/near_duplicates.csv")
if nd_path.exists():
    nd = pd.read_csv(nd_path)
    print(f"Cặp ảnh gần trùng (EDA)    : {len(nd)} | khác lesion_id: {(~nd['same_lesion']).sum()}")
else:
    print("Chưa có outputs/eda/near_duplicates.csv (chạy EDA trước) -> bỏ qua")

### 3.3 Tính nhất quán

In [ ]:
# Mỗi lesion chỉ thuộc 1 nhãn? (nếu không -> nhiễu nhãn)
assert (df.groupby("lesion_id")["dx"].nunique() == 1).all(), "Có lesion mang nhiều nhãn!"
print("Mỗi lesion_id chỉ có 1 nhãn dx.")

# Metadata trong cùng lesion có khớp không?
incons = df.groupby("lesion_id")[["dx_type", "age", "sex", "localization"]].nunique(dropna=True).gt(1).sum()
print("Số lesion có metadata không nhất quán:", incons.to_dict())
bad = df.groupby("lesion_id")["localization"].nunique().loc[lambda s: s > 1].index
df[df["lesion_id"].isin(bad)][["lesion_id", "image_id", "dx", "localization"]]

> Vài lesion có `localization` khác nhau giữa các ảnh. Đây chỉ là metadata phụ, không ảnh hưởng nhãn `dx` → giữ nguyên, ghi nhận.

In [ ]:
# Kích thước / mode của TOÀN BỘ ảnh (Image.open chỉ đọc header -> nhanh)
def img_meta(p):
    with Image.open(p) as im:
        return im.size, im.mode
sizes = pd.Series([img_meta(p) for p in df["path"]])
print(sizes.value_counts())

### 3.4 Outlier

In [ ]:
# Quy tắc IQR (Tukey): ngoài [Q1 - 1.5·IQR, Q3 + 1.5·IQR]
q1, q3 = df["age"].quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df["age_outlier"] = (df["age"] < lo) | (df["age"] > hi)
print(f"age: Q1={q1}, Q3={q3}, IQR={iqr} -> ngưỡng [{lo}, {hi}]")
print("Giá trị age khác nhau:", sorted(df["age"].dropna().unique()))
print("Số ảnh outlier theo age:", df["age_outlier"].sum(),
      "| giá trị:", df.loc[df["age_outlier"], "age"].value_counts().to_dict())
print("Phân bố dx của outlier:", df.loc[df["age_outlier"], "dx"].value_counts().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
sns.histplot(df["age"], bins=np.arange(-2.5, 90, 5), ax=axes[0], color="#7A7A7A")
axes[0].axvline(lo, ls="--", c="#D55E00", label=f"ngưỡng IQR dưới = {lo}")
axes[0].set(xlabel="Tuổi (năm)", ylabel="Số ảnh", title=f"Phân bố tuổi (n={df['age'].notna().sum()})")
axes[0].legend()
order = sorted(df["dx"].unique())
sns.boxplot(data=df, x="dx", y="age", order=order, palette=CLASS_COLORS, hue="dx", legend=False, ax=axes[1], fliersize=2)
axes[1].set(xlabel="Lớp (dx)", ylabel="Tuổi (năm)", title="Tuổi theo lớp")
plt.tight_layout(); fig.savefig(FIG_DIR / "age_outlier.pdf", bbox_inches="tight"); plt.show()

> `age` được ghi theo bội số của 5 (đã làm tròn sẵn). Các giá trị bị quy tắc IQR đánh dấu là tuổi rất thấp — vẫn **hợp lý** (trẻ em
> có nốt ruồi/tổn thương mạch máu), không phải lỗi nhập liệu. Vì `age` không phải đầu vào mô hình và ảnh/nhãn của các dòng này
> hợp lệ, **chỉ gắn cờ `age_outlier`, không xoá**. Số ảnh/lesion (tối đa 6) cũng không phải outlier cần loại: được xử lý bằng chia theo `lesion_id`.

### 3.5 Phân bố nhãn

In [ ]:
DX_NAMES = {
    "nv": "Melanocytic nevi",
    "mel": "Melanoma",
    "bkl": "Benign keratosis-like lesions",
    "bcc": "Basal cell carcinoma",
    "akiec": "Actinic keratoses / Bowen's",
    "vasc": "Vascular lesions",
    "df": "Dermatofibroma",
}
counts = df["dx"].value_counts()
print(f"Mất cân bằng: lớp lớn nhất / nhỏ nhất = {counts.max() / counts.min():.1f}x")
pd.DataFrame({"tên": counts.index.map(DX_NAMES), "số ảnh": counts,
              "%": (counts / len(df) * 100).round(2)})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
sns.countplot(data=df, x="dx", order=counts.index, hue="dx", palette=CLASS_COLORS, legend=False, ax=axes[0])
axes[0].set(xlabel="Lớp (dx)", ylabel="Số ảnh", yscale="log", title=f"Phân bố nhãn (n={len(df)}, thang log)")
sns.countplot(data=df, x="dx_type", ax=axes[1], color="#7A7A7A")
axes[1].set(xlabel="Cách chẩn đoán (dx_type)", ylabel="Số ảnh", title="dx_type")
plt.tight_layout(); fig.savefig(FIG_DIR / "class_distribution.pdf", bbox_inches="tight"); plt.show()

## 4. Mã hoá nhãn

In [ ]:
CLASSES = sorted(df["dx"].unique())
class_to_idx = {c: i for i, c in enumerate(CLASSES)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
NUM_CLASSES = len(CLASSES)
df["label"] = df["dx"].map(class_to_idx)
print(class_to_idx)

## 5. Chia train / val / test

- **Test**: dùng lại `outputs/splits/test.csv` (đã cố định cho mọi thí nghiệm). `StratifiedGroupKFold` (group = `lesion_id`,
  stratify = `label`, seed 42) được chạy lại chỉ để **kiểm tra** khớp với file đã lưu, không ghi đè.
- **Validation**: tách ~20% từ `train.csv`, cũng theo `lesion_id`, dùng để chọn epoch tốt nhất. **Test chỉ đánh giá 1 lần** ở cuối mỗi seed.

In [ ]:
n_splits = int(round(1 / TEST_SIZE))
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
train_idx, test_idx = next(sgkf.split(df, df["label"], groups=df["lesion_id"]))

cols = ["image_id", "lesion_id", "dx", "label", "path"]
if (SPLIT_DIR / "train.csv").exists() and (SPLIT_DIR / "test.csv").exists():
    saved_train = pd.read_csv(SPLIT_DIR / "train.csv")
    saved_test = pd.read_csv(SPLIT_DIR / "test.csv")
    assert set(saved_train["image_id"]) == set(df.iloc[train_idx]["image_id"]), "train.csv lệch so với split tái tạo!"
    assert set(saved_test["image_id"]) == set(df.iloc[test_idx]["image_id"]), "test.csv lệch so với split tái tạo!"
    print("Dùng lại split đã lưu (khớp với split tái tạo).")
else:
    df.iloc[train_idx][cols].to_csv(SPLIT_DIR / "train.csv", index=False)
    df.iloc[test_idx][cols].to_csv(SPLIT_DIR / "test.csv", index=False)
    saved_train = df.iloc[train_idx][cols]; saved_test = df.iloc[test_idx][cols]
    print("Chưa có split -> đã tạo và lưu.")

trainval_df = df[df["image_id"].isin(saved_train["image_id"])].reset_index(drop=True)
test_df = df[df["image_id"].isin(saved_test["image_id"])].reset_index(drop=True)

# Validation tách từ train, theo lesion_id
sgkf_val = StratifiedGroupKFold(n_splits=int(round(1 / VAL_SIZE)), shuffle=True, random_state=SEED)
fit_idx, val_idx = next(sgkf_val.split(trainval_df, trainval_df["label"], groups=trainval_df["lesion_id"]))
train_df = trainval_df.iloc[fit_idx].reset_index(drop=True)
val_df = trainval_df.iloc[val_idx].reset_index(drop=True)

for a, b in [(train_df, val_df), (train_df, test_df), (val_df, test_df)]:
    assert set(a["lesion_id"]).isdisjoint(b["lesion_id"]), "Rò rỉ lesion_id!"
train_df[cols].to_csv(SPLIT_DIR / "train_fit.csv", index=False)
val_df[cols].to_csv(SPLIT_DIR / "val.csv", index=False)
n = len(df)
print(f"train: {len(train_df)} ({len(train_df)/n:.1%}) | val: {len(val_df)} ({len(val_df)/n:.1%}) | test: {len(test_df)} ({len(test_df)/n:.1%})")

In [ ]:
dist = pd.DataFrame({f"{k}_n": v["dx"].value_counts() for k, v in
                     [("train", train_df), ("val", val_df), ("test", test_df)]}).loc[CLASSES]
for k in ["train", "val", "test"]:
    dist[f"{k}_%"] = (dist[f"{k}_n"] / dist[f"{k}_n"].sum() * 100).round(2)
dist

### 5.3 Điền giá trị thiếu (fit trên train)

`age` NaN → median của **tập train** (không dùng val/test để tính → không rò rỉ). Lưu metadata đã làm sạch kèm cột `split`
và các cờ để các phân tích sau (EDA, phân tích lỗi) dùng lại.

In [ ]:
AGE_MEDIAN = train_df["age"].median()
print("Median age (train):", AGE_MEDIAN)
for frame in (train_df, val_df, test_df):
    frame["age"] = frame["age"].fillna(AGE_MEDIAN)

meta_clean = pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")])
meta_clean.to_csv(PREP_DIR / "metadata_clean.csv", index=False)
print("Còn NaN:", meta_clean.isna().sum().sum(), "| đã lưu", PREP_DIR / "metadata_clean.csv")
meta_clean.groupby("split")[["age_missing", "age_outlier"]].sum()

## 6. Xử lý mất cân bằng lớp: Weighted Random Sampling

**Cách làm.** Mỗi ảnh train $i$ có nhãn $y_i$ được gán trọng số lấy mẫu
$$w_i = \frac{1}{n_{y_i}},$$
với $n_c$ là số ảnh của lớp $c$ trong **tập train**. `WeightedRandomSampler` rút $N = |\text{train}|$ ảnh **có hoàn lại**
với xác suất $\propto w_i$ cho mỗi epoch. Tổng trọng số mỗi lớp bằng nhau ($n_c \cdot 1/n_c = 1$) → kỳ vọng mỗi lớp chiếm
$1/7$ số mẫu trong một epoch.

**Các lựa chọn đi kèm (và lý do):**
- `CrossEntropyLoss` **không** có class weights. Lý do: WRS đã cân bằng ở mức dữ liệu; thêm class weights sẽ bù mất cân bằng *hai lần*
  (lớp hiếm vừa được rút nhiều hơn vừa có loss lớn hơn). Đây là thay đổi so với cấu hình cũ trong notebook (class weights `balanced`) → ghi lại trong `config.json`.
- Sampler chỉ dùng cho **train**. Val/test giữ phân bố tự nhiên, đánh giá bằng macro-F1 / balanced accuracy.
- Generator của sampler được seed theo từng lần chạy → tái lập được.

**Hệ quả cần lưu ý:** lớp hiếm (`df`, `vasc`) bị lặp lại nhiều lần trong một epoch (oversampling) → nguy cơ overfit trên lớp hiếm;
lớp `nv` chỉ được nhìn thấy một phần mỗi epoch (undersampling ngầm). Augmentation flip giúp các bản lặp không giống hệt nhau.

In [ ]:
class_counts = train_df["label"].value_counts().sort_index()
class_w = 1.0 / class_counts                                  # w_c = 1 / n_c
sample_weights = train_df["label"].map(class_w).to_numpy().copy()   # w_i = w_{y_i}

# Mô phỏng 1 epoch để kiểm tra sampler
g = torch.Generator().manual_seed(SEED)
sim = np.array(list(WeightedRandomSampler(sample_weights, num_samples=len(train_df), replacement=True, generator=g)))
sim_labels = train_df["label"].to_numpy()[sim]
wrs_table = pd.DataFrame({
    "n_train": class_counts.values,
    "% tự nhiên": (class_counts / class_counts.sum() * 100).round(2).values,
    "w mỗi ảnh": class_w.values,
    "số lần rút (1 epoch)": np.bincount(sim_labels, minlength=NUM_CLASSES),
    "% sau WRS": (np.bincount(sim_labels, minlength=NUM_CLASSES) / len(sim) * 100).round(2),
    "ảnh khác nhau được rút": [len(np.unique(sim[sim_labels == c])) for c in range(NUM_CLASSES)],
}, index=CLASSES)
wrs_table["lặp TB / ảnh"] = (wrs_table["số lần rút (1 epoch)"] / wrs_table["ảnh khác nhau được rút"]).round(2)
wrs_table.to_csv(OUT_DIR / "wrs_one_epoch.csv")
wrs_table

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3))
x = np.arange(NUM_CLASSES)
ax.bar(x - 0.2, wrs_table["% tự nhiên"], 0.4, label="Tự nhiên (train)", color="#BBBBBB")
ax.bar(x + 0.2, wrs_table["% sau WRS"], 0.4, label="Sau WRS (1 epoch mô phỏng)",
       color=[CLASS_COLORS[c] for c in CLASSES])
ax.axhline(100 / NUM_CLASSES, ls="--", c="k", lw=0.8, label="1/7")
ax.set(xticks=x, xticklabels=CLASSES, ylabel="% mẫu trong epoch", xlabel="Lớp (dx)",
       title=f"Phân bố lớp trước/sau Weighted Random Sampling (N={len(train_df)})")
ax.legend(); plt.tight_layout(); fig.savefig(FIG_DIR / "wrs_distribution.pdf", bbox_inches="tight"); plt.show()

## 7. Dataset & DataLoader

Baseline **không xử lý ảnh**: chỉ `Resize(224×224)` + `ToTensor` + `Normalize` (ImageNet mean/std), augmentation train = flip ngang/dọc.

Để tăng tốc (giải mã JPEG 600×450 là nút cổ chai khi `NUM_WORKERS=0`), ảnh được **resize một lần và cache trong RAM**.
Kết quả giống hệt việc resize mỗi lần vì `Resize` là phép xác định và flip giao hoán với resize.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
resize = transforms.Resize((IMG_SIZE, IMG_SIZE))

train_tf = transforms.Compose([      # áp dụng SAU resize (đã cache)
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def load_resized(paths):
    out = []
    for p in paths:
        with Image.open(p) as im:
            out.append(resize(im.convert("RGB")))
    return out

class HAM10000Dataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images, self.labels, self.transform = images, list(labels), transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

t0 = time.time()
cache = {k: load_resized(v["path"]) for k, v in [("train", train_df), ("val", val_df), ("test", test_df)]}
print(f"Cache {sum(map(len, cache.values()))} ảnh trong {time.time() - t0:.0f}s")

train_ds = HAM10000Dataset(cache["train"], train_df["label"], train_tf)
val_ds = HAM10000Dataset(cache["val"], val_df["label"], eval_tf)
test_ds = HAM10000Dataset(cache["test"], test_df["label"], eval_tf)

pin = device.type == "cuda"
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)

def make_train_loader(seed):
    # Sampler có generator riêng theo seed -> thứ tự rút tái lập được; sampler thay cho shuffle=True
    g = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_ds), replacement=True, generator=g)
    return DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=pin)

x, y = train_ds[0]
print("1 mẫu:", tuple(x.shape), "label:", y, f"({idx_to_class[y]})")

## 8. Cấu hình baseline (đóng băng)

Mọi thí nghiệm tiền xử lý ảnh sau này giữ nguyên cấu hình này, chỉ thay bước xử lý ảnh.
Chọn epoch tốt nhất theo **macro-F1 trên validation**; test chỉ được đánh giá một lần với checkpoint đó.

In [ ]:
SEEDS = [42, 43, 44]
EPOCHS = 20
LR, WEIGHT_DECAY = 1e-4, 1e-4
RETRAIN = False   # False: seed đã có kết quả trong outputs/baseline thì đọc lại thay vì train lại

CONFIG = {
    "model": "resnet18 (torchvision, IMAGENET1K_V1), fc -> 7 lớp, fine-tune toàn bộ",
    "img_size": IMG_SIZE, "resize": "Resize((224,224)) bilinear, không giữ tỉ lệ",
    "normalize": "ImageNet mean/std",
    "image_processing": "không (baseline)",
    "augmentation_train": "RandomHorizontalFlip, RandomVerticalFlip",
    "imbalance": "WeightedRandomSampler, w_i = 1/n_{y_i} (train), num_samples=|train|, replacement=True",
    "loss": "CrossEntropyLoss (không class weights)",
    "optimizer": "AdamW", "lr": LR, "weight_decay": WEIGHT_DECAY, "scheduler": "không",
    "batch_size": BATCH_SIZE, "epochs": EPOCHS,
    "model_selection": "epoch có macro-F1 cao nhất trên val",
    "seeds": SEEDS,
    "split": {"train": len(train_df), "val": len(val_df), "test": len(test_df),
              "files": ["outputs/splits/train_fit.csv", "outputs/splits/val.csv", "outputs/splits/test.csv"]},
    "missing_age": f"median train = {AGE_MEDIAN}",
}
(OUT_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding="utf-8")
pd.Series(CONFIG).to_frame("giá trị")

## 9. Huấn luyện & đánh giá

In [ ]:
def make_model():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)   # 512 -> 7 lớp
    return m.to(device)

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y); n += len(y)
    return total / n

@torch.no_grad()
def predict(model, loader, criterion):
    model.eval()
    probs, ys, total = [], [], 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        total += criterion(logits, y).item() * len(y)
        probs.append(torch.softmax(logits, 1).cpu()); ys.append(y.cpu())
    probs, ys = torch.cat(probs).numpy(), torch.cat(ys).numpy()
    return ys, probs, total / len(ys)

def compute_metrics(y_true, y_pred):
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=range(NUM_CLASSES), zero_division=0)
    m = {"accuracy": accuracy_score(y_true, y_pred),
         "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
         "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
         "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0)}
    for i, c in enumerate(CLASSES):
        m[f"precision_{c}"], m[f"recall_{c}"], m[f"f1_{c}"] = p[i], r[i], f[i]
    return m

def save_preds(frame, y, probs, path):
    out = frame[["image_id", "lesion_id", "dx", "label"]].copy()
    assert (out["label"].to_numpy() == y).all()
    out["pred"] = probs.argmax(1); out["pred_dx"] = out["pred"].map(idx_to_class)
    out["correct"] = out["pred"] == out["label"]
    for i, c in enumerate(CLASSES):
        out[f"prob_{c}"] = probs[:, i]
    out.to_csv(path, index=False)

In [ ]:
def run_seed(seed):
    run_dir = OUT_DIR / f"seed{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    seed_everything(seed)
    model = make_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    train_loader = make_train_loader(seed)

    history, best_f1, best_state, best_epoch = [], -1.0, None, -1
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        yv, pv, va_loss = predict(model, val_loader, criterion)
        mv = compute_metrics(yv, pv.argmax(1))
        history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss,
                        "val_macro_f1": mv["macro_f1"], "val_balanced_accuracy": mv["balanced_accuracy"],
                        "val_accuracy": mv["accuracy"]})
        if mv["macro_f1"] > best_f1:
            best_f1, best_epoch = mv["macro_f1"], epoch
            best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
        print(f"[seed {seed}] epoch {epoch:2d} | train_loss {tr_loss:.4f} | val_loss {va_loss:.4f} | "
              f"val_F1 {mv['macro_f1']:.4f} | val_bAcc {mv['balanced_accuracy']:.4f} | {time.time() - t0:.0f}s")
    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

    # Checkpoint tốt nhất theo val -> đánh giá val + test (test chỉ 1 lần)
    model.load_state_dict(best_state)
    torch.save(best_state, run_dir / "best_model.pt")
    yv, pv, _ = predict(model, val_loader, criterion)
    yt, pt, _ = predict(model, test_loader, criterion)
    save_preds(val_df, yv, pv, run_dir / "val_predictions.csv")
    save_preds(test_df, yt, pt, run_dir / "test_predictions.csv")
    res = {"seed": seed, "best_epoch": best_epoch,
           "val": compute_metrics(yv, pv.argmax(1)), "test": compute_metrics(yt, pt.argmax(1)),
           "test_confusion_matrix": confusion_matrix(yt, pt.argmax(1), labels=range(NUM_CLASSES)).tolist()}
    (run_dir / "metrics.json").write_text(json.dumps(res, indent=2), encoding="utf-8")
    print(f"[seed {seed}] best epoch {best_epoch} | val F1 {res['val']['macro_f1']:.4f} | "
          f"test F1 {res['test']['macro_f1']:.4f} | test bAcc {res['test']['balanced_accuracy']:.4f}")
    del model, best_state; torch.cuda.empty_cache()
    return res

results = []
for seed in SEEDS:
    f = OUT_DIR / f"seed{seed}" / "metrics.json"
    if f.exists() and not RETRAIN:
        print(f"[seed {seed}] đã có kết quả -> đọc lại {f}")
        results.append(json.loads(f.read_text(encoding="utf-8")))
    else:
        results.append(run_seed(seed))

## 10. Kết quả baseline (3 seed)

Số liệu đọc từ `outputs/baseline/seed*/metrics.json`; mean ± std (độ lệch chuẩn mẫu, ddof=1) trên 3 lần chạy.

In [ ]:
MAIN = ["macro_f1", "balanced_accuracy", "accuracy", "weighted_f1"]
rows = []
for r in results:
    for split in ("val", "test"):
        rows.append({"seed": r["seed"], "split": split, "best_epoch": r["best_epoch"], **r[split]})
per_run = pd.DataFrame(rows)
per_run.to_csv(OUT_DIR / "per_run_metrics.csv", index=False)

summary = per_run.groupby("split")[MAIN].agg(["mean", "std"])
summary.to_csv(OUT_DIR / "summary.csv")
fmt = lambda g: g[MAIN].agg(["mean", "std"]).T.apply(lambda s: f"{s['mean']:.4f} ± {s['std']:.4f}", axis=1)
display(per_run[per_run.split == "test"][["seed", "best_epoch"] + MAIN].round(4))
pd.DataFrame({s: fmt(g) for s, g in per_run.groupby("split")})

In [ ]:
# Precision / recall / F1 từng lớp trên test (mean ± std, 3 seed)
test_runs = per_run[per_run.split == "test"]
pc = pd.DataFrame({
    m: [f"{test_runs[f'{m}_{c}'].mean():.3f} ± {test_runs[f'{m}_{c}'].std():.3f}" for c in CLASSES]
    for m in ["precision", "recall", "f1"]}, index=CLASSES)
pc["n_test"] = test_df["dx"].value_counts().loc[CLASSES].values
pc_num = pd.DataFrame({f"{m}_{s}": [test_runs[f"{m}_{c}"].agg(s) for c in CLASSES]
                       for m in ["precision", "recall", "f1"] for s in ["mean", "std"]}, index=CLASSES)
pc_num.to_csv(OUT_DIR / "per_class_test.csv")
pc

In [ ]:
# Đường học: train/val loss và val macro-F1 theo epoch (chọn epoch bằng val)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3))
seed_colors = dict(zip(SEEDS, ["#0072B2", "#E69F00", "#009E73"]))
for s in SEEDS:
    h = pd.read_csv(OUT_DIR / f"seed{s}" / "history.csv")
    axes[0].plot(h.epoch, h.train_loss, c=seed_colors[s], ls="--", label=f"train, seed {s}")
    axes[0].plot(h.epoch, h.val_loss, c=seed_colors[s], label=f"val, seed {s}")
    axes[1].plot(h.epoch, h.val_macro_f1, c=seed_colors[s], label=f"seed {s}")
    be = h.val_macro_f1.idxmax()
    axes[1].scatter(h.epoch[be], h.val_macro_f1[be], c=seed_colors[s], marker="*", s=120, zorder=3)
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy loss", title="Loss (train: lấy mẫu WRS; val: phân bố tự nhiên)")
axes[1].set(xlabel="Epoch", ylabel="Macro-F1", title=f"Macro-F1 trên val (n={len(val_df)}); ★ = epoch được chọn")
axes[0].legend(ncol=2, fontsize=7); axes[1].legend()
plt.tight_layout(); fig.savefig(FIG_DIR / "learning_curves.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# Confusion matrix trên test: chuẩn hoá theo hàng (= recall), trung bình 3 seed
cms = np.array([r["test_confusion_matrix"] for r in results], dtype=float)
cm_norm = (cms / cms.sum(2, keepdims=True)).mean(0)
fig, ax = plt.subplots(figsize=(5.2, 4.4))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, xticklabels=CLASSES, yticklabels=CLASSES,
            cbar_kws={"label": "Tỉ lệ theo hàng"}, ax=ax, annot_kws={"size": 7})
ax.set(xlabel="Dự đoán", ylabel="Nhãn thật",
       title=f"Confusion matrix test (n={len(test_df)}), TB {len(SEEDS)} seed")
plt.tight_layout(); fig.savefig(FIG_DIR / "confusion_matrix_test.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# F1 từng lớp trên test: cột = mean, thanh = std, điểm = từng seed
fig, ax = plt.subplots(figsize=(6.5, 3))
x = np.arange(NUM_CLASSES)
means = [test_runs[f"f1_{c}"].mean() for c in CLASSES]
stds = [test_runs[f"f1_{c}"].std() for c in CLASSES]
ax.bar(x, means, yerr=stds, capsize=3, color=[CLASS_COLORS[c] for c in CLASSES], alpha=0.85)
for i, c in enumerate(CLASSES):
    ax.scatter(np.full(len(test_runs), i) + np.linspace(-0.12, 0.12, len(test_runs)), test_runs[f"f1_{c}"],
               c="k", s=10, zorder=3)
ax.set(xticks=x, xticklabels=[f"{c}\n(n={n})" for c, n in zip(CLASSES, pc["n_test"])], ylim=(0, 1),
       ylabel="F1 (test)", title=f"F1 từng lớp – baseline ResNet-18 + WRS ({len(SEEDS)} seed)")
plt.tight_layout(); fig.savefig(FIG_DIR / "per_class_f1_test.pdf", bbox_inches="tight"); plt.show()

## 11. Nhận xét

TODO: điền sau khi chạy (mọi số lấy từ `outputs/baseline/`).